# Нейро-аналитик конкурентов · финальное приложение

Сайт → проверка профиля → подтверждение конкурентов → сбор → проверяемый отчёт. Ни один шаг не утверждает отсутствие услуги из-за ошибки загрузки.

Шаги 1–3 проверяют код и воспроизводят сохранённые исследования двух отраслей. **Последняя ячейка запускает новый пользовательский проект через input/upload.** По умолчанию платных вызовов нет. Модель включается только отдельным явным ответом, ключ берётся из Colab Secret `PROXYAPI_KEY`.

Оригинальный комплект этапа 3 находится внутри `releases/stage_3_20261027/`, включая Word. Сохранённые снимки и свежий HTTP-сбор — разные режимы.

Код и публичная база: https://github.com/gorlanova-automation/competitor-neuro-analyst .

In [1]:
#@title 1. Загрузить зафиксированный комплект и установить зависимости
import hashlib,io,json,pathlib,sys,zipfile,subprocess,urllib.request
PACKAGE_URL='https://raw.githubusercontent.com/gorlanova-automation/competitor-neuro-analyst/main/stage_4/packages/25a7dd286ef9618792c9045fb092060fb4d6798195fcc3c8bb3e5aab95aabfda.zip'
with urllib.request.urlopen(PACKAGE_URL,timeout=60) as response:payload=response.read(15000000)
assert hashlib.sha256(payload).hexdigest()=="25a7dd286ef9618792c9045fb092060fb4d6798195fcc3c8bb3e5aab95aabfda", 'Package checksum mismatch'
WORK=pathlib.Path.cwd()/'neuro_analyst_final'
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        if not (WORK/name).resolve().is_relative_to(WORK.resolve()):raise ValueError('Unsafe archive path')
    archive.extractall(WORK)
subprocess.check_call([sys.executable,'-m','pip','-q','install','-r',str(WORK/'stage_3/code/requirements.txt'),'-r',str(WORK/'app/requirements.txt')])
sys.path.insert(0,str(WORK));sys.path.insert(0,str(WORK/'stage_3/code'))
print('Комплект распакован; API-ключ не запрашивался.')


Комплект распакован; API-ключ не запрашивался.


In [2]:
# 2. Проверить движок, приложение, подтверждения и модельный контракт без платных вызовов
import unittest
suite=unittest.defaultTestLoader.discover(str(WORK/'stage_3/code'),pattern='test_*.py')
for module in ['stage_4.test_workflow','stage_4.test_semantic','stage_4.test_review','app.test_api','app.test_workflow_api']:
    suite.addTests(unittest.defaultTestLoader.loadTestsFromName(module))
result=unittest.TextTestRunner(verbosity=1,stream=sys.stdout).run(suite)
assert result.wasSuccessful()
manifest=json.loads((WORK/'releases/stage_3_20261027/manifest.json').read_text(encoding='utf-8'))
for item in manifest['files']:
    assert hashlib.sha256((WORK/'releases/stage_3_20261027'/item['file']).read_bytes()).hexdigest()==item['sha256']
print('PASS:',result.testsRun,'тестов. Этап 3: все контрольные суммы совпали. Новых LLM-вызовов: 0.')


...........................................................
----------------------------------------------------------------------
Ran 59 tests in 2.114s

OK
PASS: 59 тестов. Этап 3: все контрольные суммы совпали. Новых LLM-вызовов: 0.


In [3]:
# 3. Две отрасли на сохранённых реальных текстах — не свежий сбор
from stage_4.colab_app import run_saved_demo
for case in ['immigration','shift']:
    state=run_saved_demo(project_root=WORK,output_dir=WORK/'var/demo_projects',case=case)
    print(case,state['status'],state['report'])
print('Подтверждения в этих двух прогонах демонстрационные. Они не заменяют решения пользователя.')
print('Новых LLM-вызовов: 0. Следующая ячейка — ваш новый проект.')


immigration report_ready {'run_id': '7720cedd-bf54-4a3d-b91a-da6dd0ac77d3', 'companies': 2, 'companies_without_content': 0, 'evidence': 41, 'new_llm_calls': 0, 'api_cost': 0}
shift report_ready {'run_id': '281ae6e8-dad7-4b21-ae7d-998a2ab04a53', 'companies': 2, 'companies_without_content': 0, 'evidence': 20, 'new_llm_calls': 0, 'api_cost': 0}
Подтверждения в этих двух прогонах демонстрационные. Они не заменяют решения пользователя.
Новых LLM-вызовов: 0. Следующая ячейка — ваш новый проект.


## Запуск своего исследования

Нужны URL своей компании, описание рынка и 1–20 подтверждённых конкурентов. Можно попробовать автоматический поиск; если он недоступен, укажите адреса вручную. Файлы новых результатов пишутся в отдельные папки.

Смысловой ответ модели — черновик с проверенными цитатами. Проверка цитат не гарантирует правильность интерпретации. Instagram/Facebook и визуал не объявляются изученными по найденным ссылкам. Для обхода закрытых страниц и CAPTCHA инструментов нет.

**Платный шаг необязателен:** один запрос ProxyAPI, максимум 2500 токенов ответа, без автоматического повтора. Не вставляйте ключ в текст ячеек.

In [ ]:
# 4. Пользовательский запуск: ввод данных / загрузка TXT
from stage_4.colab_app import launch_colab
user_project=launch_colab(output_dir=WORK/'var/projects')
